# Annotate merged single cells with metadata from platemap file

## Import libraries

In [1]:
import argparse
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tqdm
from pycytominer import annotate
from pycytominer.cyto_utils import output

try:
    cfg = get_ipython().config
    in_notebook = True
except NameError:
    in_notebook = False

## Set paths and variables

In [2]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path("../../data/").resolve()

# directory where parquet files are located
data_dir = pathlib.Path("../data/1.annotated_data").resolve()

# directory where the annotated parquet files are saved to
profiles_output_dir = pathlib.Path(
    "../data/2.sc_tracks_annotated_data/profiles/timelapse/"
).resolve()
stats_output_dir = pathlib.Path(
    "../data/2.sc_tracks_annotated_data/stats/timelapse"
).resolve()

profiles_output_dir.mkdir(exist_ok=True, parents=True)
stats_output_dir.mkdir(exist_ok=True, parents=True)

if not in_notebook:
    print("Running as script")
    # set up arg parser
    parser = argparse.ArgumentParser(description="Single cell extraction")

    parser.add_argument(
        "--well_fov",
        type=str,
        help="Path to the input directory containing the tiff images",
    )

    args = parser.parse_args()
    well_fov = args.well_fov
else:
    print("Running in a notebook")
    well_fov = "C-02_F0003"

Running in a notebook


In [3]:
tracks = pathlib.Path(
    f"../../5.cell_tracking/results/{well_fov}_tracks.parquet"
).resolve(strict=True)
profiles = pathlib.Path(
    f"../data/1.annotated_data/timelapse/{well_fov}_sc.parquet"
).resolve(strict=True)

tracks = pd.read_parquet(tracks)
profiles = pd.read_parquet(
    profiles,
)
# prepend Metadata_ to the tracks columns
tracks.columns = ["Metadata_" + str(col) for col in tracks.columns]
tracks["Metadata_coordinates"] = list(zip(tracks["Metadata_x"], tracks["Metadata_y"]))
profiles["Metadata_coordinates"] = list(
    zip(profiles["Nuclei_AreaShape_Center_X"], profiles["Nuclei_AreaShape_Center_Y"])
)

profiles["Metadata_Time"] = profiles["Metadata_Time"].astype(float)
profiles["Metadata_Time"] = profiles["Metadata_Time"] - 1

In [4]:
coordinate_column_left = "Metadata_coordinates"
coordinate_column_right = "Metadata_coordinates"
pixel_cutt_off = 5
left_on = ["Metadata_Time"]
right_on = ["Metadata_t"]
merged_df_list = []  # list to store the merged dataframes
total_CP_cells = 0  # total number of cells in the left dataframe
total_annotated_cells = 0  # total number of cells that were annotated
distances = []  # list to store the distances between the coordinates

In [5]:
tracked_cells_stats = {
    "Metadata_time": [],  # timepoint of the cell
    "total_CP_cells": [],  # total number of cells segmented
    "total_annotated_cells": [],  # total number of cells tracked
}
for time in profiles["Metadata_Time"].unique():
    df_left = profiles.copy().loc[profiles["Metadata_Time"] == time]
    df_right = tracks.copy().loc[tracks["Metadata_t"] == time]

    total_CP_cells += df_left.shape[0]

    # loop through the rows in the subset_annotated_df and find the closest coordinate set in the location metadata
    for index1, row1 in df_left.iterrows():
        # appends 1 for the total number of cells segmented
        # after the loop, the total number of cells segmented is the sum of all the 1s in the list
        tracked_cells_stats["total_CP_cells"].append(1)
        dist = np.inf
        for index2, row2 in df_right.iterrows():
            coord1 = row1[coordinate_column_left]
            coord2 = row2[coordinate_column_right]
            try:
                temp_dist = np.linalg.norm(np.array(coord1) - np.array(coord2))
            except:
                temp_dist = np.inf
            if temp_dist <= dist:
                dist = temp_dist
                coord2_index = index2

            # set cut off of 5,5 pixel in the euclidean distance
            euclidean_cut_off = np.linalg.norm(
                np.array([0, 0]) - np.array([pixel_cutt_off, pixel_cutt_off])
            )

        if dist < euclidean_cut_off:
            temp_merged_df = pd.merge(
                df_left.loc[[index1]],
                df_right.loc[[coord2_index]],
                how="inner",
                left_on=left_on,
                right_on=right_on,
            )
            distances.append(dist)
            total_annotated_cells += temp_merged_df.shape[0]
            tracked_cells_stats["Metadata_time"].append(time)
            # if the cell is tracked and annotated, append 1 to the total number of cells annotated
            tracked_cells_stats["total_annotated_cells"].append(1)
            merged_df_list.append(temp_merged_df)
        else:
            tracked_cells_stats["Metadata_time"].append(time)
            # if the cell is not tracked and annotated, append 0 to the total number of cells annotated
            tracked_cells_stats["total_annotated_cells"].append(0)
if len(merged_df_list) == 0:
    merged_df_list.append(pd.DataFrame())
merged_df = pd.concat(merged_df_list)
merged_df["Metadata_distance"] = distances

# replace Metadata string in column names with Metadata (Non Morphology Features)
merged_df.columns = [
    x.replace("Metadata_", "Metadata_") if "Metadata_" in x else x
    for x in merged_df.columns
]

print(f"Annotated cells: {total_annotated_cells} out of {total_CP_cells}")
print(f"Percentage of annotated cells: {total_annotated_cells/total_CP_cells*100}%")
print(merged_df.shape)
merged_df.to_parquet(profiles_output_dir / f"{well_fov}_annotated_tracks.parquet")
merged_df.head()

Annotated cells: 1833 out of 2279
Percentage of annotated cells: 80.43001316366828%
(1833, 2335)


,Metadata_plate,Metadata_Well,Metadata_number_of_singlecells,Metadata_compound,Metadata_dose,Metadata_control,Metadata_ImageNumber,Metadata_FOV,Metadata_Time,Metadata_Cells_Number_Object_Number,...,Metadata_coordinates_x,Metadata_track_id,Metadata_t,Metadata_y,Metadata_x,Metadata_id,Metadata_parent_track_id,Metadata_parent_id,Metadata_coordinates_y,Metadata_distance
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,2,...,"(341.81398601398604, 7.676923076923077)",1,0.0,8.0,342.0,1000002.0,-1,-1.0,"(342.0, 8.0)",0.372800
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,4,...,"(733.322869955157, 10.36883408071749)",2,0.0,10.0,733.0,1000004.0,-1,-1.0,"(733.0, 10.0)",0.490187
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,18,...,"(1113.3801756587202, 89.93350062735257)",3,0.0,90.0,1113.0,1000017.0,-1,-1.0,"(1113.0, 90.0)",0.385948
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,20,...,"(1006.3214285714286, 91.00714285714285)",4,0.0,91.0,1006.0,1000019.0,-1,-1.0,"(1006.0, 91.0)",0.321508
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,24,...,"(454.9419113054341, 125.83510306058713)",5,0.0,126.0,455.0,1000023.0,-1,-1.0,"(455.0, 126.0)",0.174829


In [6]:
# get the number of tracks for each track length
list_of_track_lengths = []
for track in merged_df["Metadata_track_id"].unique():
    track_length = merged_df.loc[merged_df["Metadata_track_id"] == track].shape[0]
    list_of_track_lengths.append(track_length)
list_of_track_lengths_df = pd.DataFrame(list_of_track_lengths, columns=["track_length"])
list_of_track_lengths_df = (
    list_of_track_lengths_df.value_counts().to_frame().reset_index()
)
list_of_track_lengths_df["well_fov"] = well_fov
# save the list of track lengths to a parquet file
list_of_track_lengths_df.to_parquet(
    stats_output_dir / f"{well_fov}_track_lengths.parquet"
)

In [7]:
# save the tracked cells stats to a parquet file
tracked_cells_stats_df = pd.DataFrame(tracked_cells_stats)
tracked_cells_stats_df["well_fov"] = well_fov
tracked_cells_stats_df
# get the number of cells for each time point
tracked_cells_stats_df = (
    tracked_cells_stats_df.groupby(["Metadata_time", "well_fov"]).sum().reset_index()
)

In [8]:
# save the stats to a parquet file
tracked_cells_stats_df.to_parquet(stats_output_dir / f"{well_fov}_stats.parquet")

In [9]:
tracked_cells_stats_df

,Metadata_time,well_fov,total_CP_cells,total_annotated_cells
0,0.0,C-02_F0003,183,32
1,1.0,C-02_F0003,181,140
2,2.0,C-02_F0003,184,144
3,3.0,C-02_F0003,181,146
4,4.0,C-02_F0003,176,150
5,5.0,C-02_F0003,171,152
6,6.0,C-02_F0003,174,154
7,7.0,C-02_F0003,172,152
8,8.0,C-02_F0003,171,148
9,9.0,C-02_F0003,169,152


In [10]:
merged_df

,Metadata_plate,Metadata_Well,Metadata_number_of_singlecells,Metadata_compound,Metadata_dose,Metadata_control,Metadata_ImageNumber,Metadata_FOV,Metadata_Time,Metadata_Cells_Number_Object_Number,...,Metadata_coordinates_x,Metadata_track_id,Metadata_t,Metadata_y,Metadata_x,Metadata_id,Metadata_parent_track_id,Metadata_parent_id,Metadata_coordinates_y,Metadata_distance
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,2,...,"(341.81398601398604, 7.676923076923077)",1,0.0,8.0,342.0,1000002.0,-1,-1.0,"(342.0, 8.0)",0.372800
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,4,...,"(733.322869955157, 10.36883408071749)",2,0.0,10.0,733.0,1000004.0,-1,-1.0,"(733.0, 10.0)",0.490187
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,18,...,"(1113.3801756587202, 89.93350062735257)",3,0.0,90.0,1113.0,1000017.0,-1,-1.0,"(1113.0, 90.0)",0.385948
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,20,...,"(1006.3214285714286, 91.00714285714285)",4,0.0,91.0,1006.0,1000019.0,-1,-1.0,"(1006.0, 91.0)",0.321508
0,1,C-02,183,Staurosporine,0.0,negative,1,0003,0.0,24,...,"(454.9419113054341, 125.83510306058713)",5,0.0,126.0,455.0,1000023.0,-1,-1.0,"(455.0, 126.0)",0.174829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,1,C-02,172,Staurosporine,0.0,negative,8,0003,7.0,165,...,"(1774.7553444180523, 1772.0118764845606)",151,7.0,1772.0,1775.0,8000170.0,-1,7000167.0,"(1775.0, 1772.0)",0.244944
0,1,C-02,172,Staurosporine,0.0,negative,8,0003,7.0,167,...,"(71.41873669268985, 1836.8257629524485)",153,7.0,1837.0,71.0,8000171.0,-1,7000169.0,"(71.0, 1837.0)",0.453540
0,1,C-02,172,Staurosporine,0.0,negative,8,0003,7.0,168,...,"(380.8662891986063, 1855.4316202090592)",152,7.0,1855.0,381.0,8000172.0,-1,7000168.0,"(381.0, 1855.0)",0.451857
0,1,C-02,172,Staurosporine,0.0,negative,8,0003,7.0,169,...,"(1114.8011472275334, 1876.0905035054175)",154,7.0,1876.0,1115.0,8000173.0,-1,7000171.0,"(1115.0, 1876.0)",0.218480
